# CNN Deep SVDD — Train from Scratch

Pipeline: Generate synthetic images → Extract patches → Initialize center **c** →
Deep SVDD train (minimise ‖f(x) − c‖²) → Compute R² → Save weights.

All code is self-contained — no dependency on the `padi` package.

In [1]:
import sys, os
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# Ensure the repo root is on sys.path so pythonsi is importable
REPO = Path.cwd()
while REPO.name and not (REPO / "pythonsi").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from network import PatchNetwork

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cpu


In [ ]:
# --- CONFIGURATION ---
SEED = 42
PATCH_SZ = 15
PATCH_STR = 15  # non-overlapping patches
IMG_SIZE = (300, 300)  # full image size (H, W)
N_TRAIN = 500  # number of training images
REPDIM = 16
CHANNELS = (8, 16)  # two conv blocks → 2 MaxPool
BATCH_SIZE = 64
SVDD_EPOCHS = 60


torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Generate Synthetic Training Images

Normal images: each pixel follows N(0, 1).
Anomaly images: each pixel follows N(delta, 1).
No normalization or clamping needed — pixel variance is 1.0.

In [3]:
def generate_normal_images(n_samples, img_size, channels=1):
    """Normal images: each pixel ~ N(0, 1)."""
    h, w = img_size
    imgs = np.random.normal(loc=0.0, scale=1.0, size=(n_samples, channels, h, w))
    return torch.from_numpy(imgs.astype(np.float32))


def extract_patches(img_tensor, patch_size, stride):
    """Extract patches from a (B, C, H, W) tensor.

    Returns (B * n_h * n_w, C, patch_size, patch_size).
    """
    if img_tensor.dim() == 3:
        img_tensor = img_tensor.unsqueeze(0)
    patches = img_tensor.unfold(2, patch_size, stride).unfold(3, patch_size, stride)
    B, C, n_h, n_w, pH, pW = patches.shape
    return patches.contiguous().view(B * n_h * n_w, C, pH, pW)


train_images = generate_normal_images(N_TRAIN, IMG_SIZE)
print(f"Train images: {train_images.shape}")

train_patches = extract_patches(train_images, PATCH_SZ, PATCH_STR)
print(f"Train patches: {train_patches.shape}")
print(f"Patches per image: {train_patches.shape[0] // N_TRAIN}")

Train images: torch.Size([500, 1, 300, 300])
Train patches: torch.Size([50000, 1, 30, 30])
Patches per image: 100


## 2. Initialize Center **c** & Train Deep SVDD

1. Build PatchNetwork encoder.
2. Compute center **c** = mean of encoder outputs over training set.
3. Minimise ‖f(x) − c‖².

In [ ]:
net = PatchNetwork(
    in_channels=1,
    img_size=(PATCH_SZ, PATCH_SZ),
    repdim=REPDIM,
    channels=CHANNELS,
).to(DEVICE)

train_dataset = TensorDataset(train_patches)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- Initialize center c = mean of encoder outputs ---
net.eval()
all_outputs = []
with torch.no_grad():
    for (batch,) in DataLoader(train_dataset, batch_size=BATCH_SIZE):
        all_outputs.append(net(batch.to(DEVICE)).cpu())
all_outputs = torch.cat(all_outputs, dim=0)
center_c = all_outputs.mean(dim=0).to(DEVICE)
print(f"Center c shape: {center_c.shape}")

# --- Deep SVDD training ---
net.train()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-6)

print("\nDeep SVDD training …")
for epoch in range(SVDD_EPOCHS):
    total_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(DEVICE)
        outputs = net(batch)
        dist = torch.sum((outputs - center_c) ** 2, dim=1)
        loss = torch.mean(dist)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(
            f"  Epoch {epoch + 1}/{SVDD_EPOCHS} | Loss: {total_loss / len(train_loader):.8f}"
        )

print("Deep SVDD training done.")

Center c shape: torch.Size([16])

Deep SVDD training …
  Epoch 1/100 | Loss: 0.23313450


## 3. Compute R² and Save Weights

In [ ]:
net.eval()
all_dists = []
with torch.no_grad():
    for (batch,) in DataLoader(train_dataset, batch_size=BATCH_SIZE):
        outputs = net(batch.to(DEVICE))
        dist = torch.sum((outputs - center_c) ** 2, dim=1)
        all_dists.append(dist.cpu().numpy())

all_dists = np.concatenate(all_dists)
R_squared = all_dists
print(f"R² = {R_squared:.10f}")
print(f"Center c (first 4): {center_c[:4].cpu().numpy()}")

# Save
os.makedirs("weights", exist_ok=True)
torch.save(
    {
        "model_state_dict": net.state_dict(),
        "center_c": center_c.cpu().numpy(),
        "R_squared": R_squared,
        "config": {
            "in_channels": 1,
            "img_size": (PATCH_SZ, PATCH_SZ),
            "repdim": REPDIM,
            "channels": CHANNELS,
        },
    },
    "weights/patch_network.pth",
)
print("Weights saved to weights/patch_network.pth")